# S13.3 - QLoRA Financial Instruction Tuning

This notebook mirrors the LoRA workflow, but loads the frozen base model in 4-bit precision with bitsandbytes. QLoRA is most useful when the full model does not fit comfortably in 16-bit precision.

Colab GPU setup:

```bash
pip install -q transformers datasets accelerate peft bitsandbytes
```

On CPU-only machines, run S13.2 instead. bitsandbytes 4-bit training is intended for CUDA GPUs.

In [ ]:
from pathlib import Path
import sys

def find_finance_dir(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        candidate = path / "help-code" / "4-nlp-finance"
        if candidate.exists():
            return candidate
        if path.name == "4-nlp-finance":
            return path
    raise FileNotFoundError("Run this notebook from the course repo or help-code/4-nlp-finance.")

FINANCE_DIR = find_finance_dir()
SRC_DIR = FINANCE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from finance_nlp_utils import load_evaluation_csv

rows = load_evaluation_csv("apple_2025_sentence_classification.csv")
labels = sorted({row["label"] for row in rows})
labels

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("QLoRA requires a CUDA GPU for this teaching notebook. Use S13.2 for CPU LoRA.")

torch.cuda.get_device_name(0)

In [ ]:
def format_example(sentence: str, label: str | None = None) -> str:
    label_text = "" if label is None else label
    return f"""### Instruction:
Classify the Apple 10-K disclosure sentence into one of these labels:
{', '.join(labels)}.

### Input:
{sentence}

### Response:
{label_text}"""

print(format_example(rows[0]["sentence"], rows[0]["label"]))

In [ ]:
from datasets import Dataset, DatasetDict

splits = {}
for split in ["train", "validation", "test"]:
    examples = [{"text": format_example(row["sentence"], row["label"])} for row in rows if row["split"] == split]
    splits[split] = Dataset.from_list(examples)
dataset = DatasetDict(splits)
dataset

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, qlora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize(batch):
    tokens = tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized.set_format("torch")
tokenized

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=str(FINANCE_DIR / "outputs" / "sec-risk-qlora"),
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    bf16=True,
    optim="paged_adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
)
trainer.train()
model.save_pretrained(FINANCE_DIR / "outputs" / "sec-risk-qlora-adapter")

In [ ]:
prompt = format_example("The Company may be affected by unfavorable macroeconomic conditions.", None)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(generated[0], skip_special_tokens=True))